In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

#################################
# 1. Config & Paths 
#################################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_path = "C:/Users/user/Desktop/IDS_masters/dataset/audi_robust_0305.npz"
model1_path = "C:/Users/user/Desktop/IDS_masters/model/TCN1_{now}.pth"
model2_path = "C:/Users/user/Desktop/IDS_masters/model/TCN2_{now}.pth"

batch_size = 64
Stage1_CH = [0, 1, 2, 3, 4, 5]
Stage2_CH = [4, 9]

num_input1 = 6
num_input2 = 2

In [2]:
#################################
# 2. Dataset Load Class
#################################
class LoadDatset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.from_numpy(tensor_X).float()
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.from_numpy(tensor_y).long()

        self.X = torch.nan_to_num(tensor_X, nan=0.0)
        self.y = tensor_y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

#################################
# 3. Causal Convolution Layer
#################################
class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0
        )
        self.left_pad = (kernel_size - 1) * dilation

    def forward(self, x):
        x = F.pad(x, (self.left_pad, 0))
        return super().forward(x)

#################################
# 4. TCN Blocks
#################################
class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernal_size=3,  dilation=1, dropout=0.1):
        super().__init__()

        self.conv1 = CausalConv1d(n_inputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.conv3 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)
        
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        def init_one(layer):
            w = getattr(layer, "weight_orig", None)
            if w is None:
                w = layer.weight
            nn.init.kaiming_normal_(w)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

        init_one(self.conv1)
        init_one(self.conv2)
        init_one(self.conv3)

        if self.downsample is not None:
            nn.init.kaiming_normal_(self.downsample.weight)
            if self.downsample.bias is not None:
                nn.init.zeros_(self.downsample.bias)
                
    def forward(self, x):
        out = self.conv1(x)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        out = self.conv3(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channel, kernel_size=3, dropout =0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channel)
        dilation = [1,2,4]

        for i in range(num_levels):
            dilation_size = dilation[i]
            in_channels = num_inputs if i == 0 else num_channel[i-1]
            out_channels = num_channel[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size,  dilation = dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

#################################
# 5. Model Architecture (SeqIDS)
#################################
class SeqIDS(nn.Module):
    def __init__(self, num_input, num_classes, dropout_rate=0.2):
        super(SeqIDS, self).__init__()

        self.tcn = TemporalConvNet(
            num_inputs=num_input,
            num_channel=[32, 64, 128],
            kernel_size=3,
            dropout=dropout_rate
        )

        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Conv1d(128, num_classes, kernel_size=1)

    def forward(self, x, return_attn=False):
        x = self.tcn(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        if return_attn:
            return logits
        
        return logits

#################################
# 8.  Multi TCN (Evaluation)
#################################

model1 = SeqIDS(num_input=num_input1, num_classes=3, dropout_rate=0.5).to(device)
model2 = SeqIDS(num_input=num_input2, num_classes=2, dropout_rate=0.5).to(device)

# 저장된 weight 로드
state1 = torch.load((model1_path), map_location=device)
model1.load_state_dict(state1)

state2 = torch.load((model2_path), map_location=device)
model2.load_state_dict(state2)

model1.eval()
model2.eval()

test_data = np.load(test_path)
X_np, y_np = test_data["X"], test_data["y"]

test_ds = LoadDatset(X_np, y_np)
test_loader = DataLoader(
    test_ds,
    batch_size,
    shuffle=False
)

correct = 0
total = 0
th_dos = 0.9
th_fuzz = 0.2

num_classes = 5
conf_mat = torch.zeros(num_classes, num_classes, dtype=torch.int64)

with torch.no_grad():
    for input, labels in test_loader:
        input = input.to(device)
        # input = input.permute(0,2,1)
        labels = labels.to(device)

        B, C, L = input.shape

        x1 = input[:, Stage1_CH, :]
        
        ##============= Stage1 ================##
        logit1 = model1(x1)
        p1 = torch.softmax(logit1, dim=1)

        logit_dos = logit1[:,1,:]
        logit_fuzz = logit1[:,2,:]

        dos_mask = p1[:,1,:] >= th_dos
        fuzz_mask = p1[:,2,:] >= th_fuzz

        pred = torch.zeros((B,L), dtype=torch.long, device=device)

        ### dos만 임계값 초과 ###
        only_dos = dos_mask & ~fuzz_mask
        pred[only_dos] = 1

        ### fuzzing만 임계값 초과 ###
        only_fuzz = fuzz_mask & ~dos_mask
        pred[only_fuzz] = 2

        ### dos, fuzzing 모두 임계값 초과 할 때 logit 큰 쪽으로 결과값 ###
        both = dos_mask & fuzz_mask
        pred[both] = torch.where(
            logit_dos[both] >= logit_fuzz[both],
            torch.ones_like(logit_dos[both], dtype=torch.long),   # DoS = 1
            torch.full_like(logit_dos[both], 2, dtype=torch.long) # Fuzz = 2
        )

        ### Stage2로 보낼 위치 ###
        decided = dos_mask | fuzz_mask

        ##============= Stage2 ================##
        x2 = input[:, Stage2_CH, :]
        logit2 = model2(x2)
        pred2 = logit2.argmax(dim=1)

        mapped2 = pred2.clone()
        mapped2[pred2 == 0] = 0 
        mapped2[pred2 == 1] = 4

        pred[~decided] = mapped2[~decided]

        ##============= Metrics =============##
        t_flat = labels.reshape(-1).cpu()
        p_flat = pred.reshape(-1).cpu()

        total += t_flat.numel()
        correct += (t_flat == p_flat).sum().item()

        for t, p in zip(t_flat, p_flat):
            conf_mat[t.long(), p.long()] += 1

accuracy = correct / total

row_sum = conf_mat.sum(dim=1)
tp = conf_mat.diag()
fp = conf_mat.sum(dim=0) - tp
fn = row_sum - tp

precision_per_class = tp / (tp + fp + 1e-12)
recall_per_class    = tp / (tp + fn + 1e-12)
f1_per_class        = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + 1e-12)

present = row_sum > 0
precision_macro = precision_per_class[present].mean().item()
recall_macro    = recall_per_class[present].mean().item()
f1_macro        = f1_per_class[present].mean().item()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision(macro, present only): {precision_macro:.4f}")
print(f"Recall(macro, present only)   : {recall_macro:.4f}")
print(f"F1(macro, present only)       : {f1_macro:.4f}")
print("Confusion Matrix:")
print(conf_mat)


EVAL_CLASSES = [0, 1, 2, 4]
eval_idx = torch.tensor(EVAL_CLASSES)

precision_macro = precision_per_class[eval_idx].mean().item()
recall_macro    = recall_per_class[eval_idx].mean().item()
f1_macro        = f1_per_class[eval_idx].mean().item()

LABEL_NAME = {0:"Normal",1:"Dos",2:"Fuzzing",4:"Spoofing"}

print("\n=== Per-class (Attack) Performance ===")
for i in EVAL_CLASSES:
    total_i = int(row_sum[i].item())
    correct_i = int(tp[i].item())
    acc_i = 100.0 * correct_i / total_i if total_i > 0 else 0.0
    print(f"{LABEL_NAME[i]:>10s} : {acc_i:6.2f}%  (correct {correct_i}/{total_i})")

C:\Users\user\AppData\Local\Temp\ipykernel_25216\3053244477.py:147: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state1 = torch.load((model1_path), map_location=device)
C:\

Accuracy : 0.9601
Precision(macro, present only): 0.8624
Recall(macro, present only)   : 0.9627
F1(macro, present only)       : 0.8984
Confusion Matrix:
tensor([[834590,    252,  32039,      0,  12543],
        [     0, 338425,      0,      0,      0],
        [   795,      0,  41655,      0,     16],
        [     0,      0,      0,      0,      0],
        [  9365,      0,     29,      0, 109619]])

=== Per-class (Attack) Performance ===
    Normal :  94.90%  (correct 834590/879424)
       Dos : 100.00%  (correct 338425/338425)
   Fuzzing :  98.09%  (correct 41655/42466)
  Spoofing :  92.11%  (correct 109619/119013)
